# PDF Keyword + Appeal-Scope Harvester (ET Corpus → _Matches)

This script scans a large folder of ET-case PDFs and builds a **targeted "matches" library** for manual review and downstream analysis.

---

## Purpose

Turn a large, unstructured PDF corpus into a structured shortlist of potentially relevant cases using deterministic keyword and regex filtering.

This is a **high-speed recall harvester**, not a semantic or LLM-based precision layer.

---

## Step 1 — Crawl and Pre-Filter PDFs

* Recursively finds all `*.pdf` files under `INPUT_ROOT`.
* Rejects small documents (`pages < MIN_PAGES`).
* Extracts text from the first `TEXT_PAGES_TO_SCAN` pages (cheap front-scan).

This keeps processing fast while avoiding irrelevant small files.

---

## Step 2 — Two-Tier Matching Logic

### Gate Condition (Mandatory)

Every PDF must contain **all phrases in `NEEDLES_ALL`**.

If this fails → the document is ignored.

### Keep Condition (At Least One Required)

After passing the gate, the PDF must satisfy **at least one** of the following:

* Contain any phrase from `NEEDLES_ANY` (simple substring match), OR
* Match `APPEAL_SCOPE_REGEX` (detects appeal-scope limitation language such as:

  * "not raised in the appeal"
  * "outside the scope of the appeal"
  * "declined to consider"
  * "failed to engage with"
  * etc.)

Only documents passing Gate + Keep are retained.

---

## Step 3 — Parallel Scanning

* Uses `ProcessPoolExecutor` with up to `MAX_WORKERS` processes.
* Each PDF is scanned independently.

For matched files, metadata collected includes:

* `path`
* `pages`
* `size_mb`
* `mtime`
* `hit_all`
* `hit_any`
* `appeal_scope_hit`
* `appeal_scope_match` (small snippet of matched phrase)

---

## Step 4 — Structured Copy to Matches Folder

All matched PDFs are copied into `MATCHES_ROOT`.

### Folder Structure

* One folder per `NEEDLES_ANY` term (slugified)
* One dedicated folder `_APPEAL_SCOPE_REGEX` for all regex hits

If `PRESERVE_STRUCTURE = True`, original directory hierarchy is preserved under each subfolder.

This prevents filename collisions and preserves provenance.

---

## Step 5 — Master Index CSV

Exactly one CSV is written to:

`MATCHES_ROOT/_matches_index.csv`

The CSV contains:

* File metadata
* Raw hit indicators
* Boolean columns per NEEDLES_ANY (e.g., `has__predetermination`)
* Boolean column for regex hits (`has__appeal_scope_regex`)
* Root path reference

---

## Output Artifacts

1. Curated PDF library under `MATCHES_ROOT/`
2. Single master index CSV for analysis

---

## Architectural Position

This module is a **deterministic recall harvester**.

It does not perform:

* Semantic analysis
* LLM reasoning
* Paragraph-level anchoring

Its role is to reduce a massive PDF corpus into a manageable, topic-filtered working set for deeper analysis (e.g., Moltie or WBerious precision layers).

---

## Guiding Principle

Fast recall first. Precision and reasoning later.


In [1]:
import shutil
import re
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from collections import Counter

import pandas as pd
import fitz  # PyMuPDF
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

INPUT_ROOT = Path(r"/media/hello/Vault/Tribunals/ET_Cases/").resolve()

# Principal matches folder (many subfolders live under here)
MATCHES_ROOT = Path(r"/media/hello/Vault/Tribunals/_Matches").resolve()

# Gate: ALL must be present (cheap recall gate)
NEEDLES_ALL = [
    "unfair dismissal",
]

# Any-of substring needles (folder buckets + boolean cols)
NEEDLES_ANY = [
    "upheld",
    "verbal warning",
    "no contemporaneous evidence",
    "predetermination",
]

APPEAL_SCOPE_REGEX = re.compile(
    r"""
    (
        (not\s+raised\s+(?:in|within)\s+the\s+(?:written\s+)?appeal) |
        (outside\s+the\s+scope\s+of\s+the\s+appeal) |
        (declined\s+to\s+consider) |
        (refused\s+to\s+consider) |
        (limited\s+to\s+the\s+grounds) |
        (confined\s+to\s+(?:the\s+)?grounds) |
        (new\s+grounds\s+(?:raised|introduced)\s+(?:at|during)\s+the\s+appeal) |
        (raised\s+at\s+the\s+appeal\s+hearing) |
        (fresh\s+consideration) |
        (rubber\s+stamp) |
        (failed\s+to\s+engage\s+with) |
        (did\s+not\s+address\s+the\s+appeal\s+ground)
    )
    """,
    re.IGNORECASE | re.VERBOSE
)

ASSUMED_INTENTION_REGEX = re.compile(
    r"""
    (
        (assum(?:e|ed|ing)\s+(?:that\s+)?(?:the\s+)?(?:claimant|employee|appellant|respondent|he|she|they|you)\s+(?:had\s+)?(?:an?\s+)?intention) |
        (assum(?:e|ed|ing)\s+(?:the\s+)?(?:claimant|employee|appellant|respondent|he|she|they|you)\s+(?:was|were)\s+(?:intend(?:ing)?|trying)\s+to) |

        ((?:his|her|their|the)\s+intention\s+(?:was|had\s+been)\s+to) |
        (intend(?:ed|ing)?\s+to\s+(?:avoid|evade|get\s+out\s+of|circumvent)) |

        (motive\s+(?:was|had\s+been)\s+to) |
        (ulterior\s+motive) |
        (improper\s+motive) |

        (purpose\s+(?:was|had\s+been)\s+to) |
        (designed\s+to\s+(?:avoid|evade|circumvent)) |
        (with\s+the\s+(?:aim|objective|intention)\s+of) |

        (pretext) |
        (a\s+sham) |
        (smokescreen) |
        (cover\s+(?:story|for)) |

        (it\s+(?:can|could|may|might)\s+be\s+inferred\s+that) |
        (i\s+infer\s+that) |
        (the\s+tribunal\s+infers\s+that)
    )
    """,
    re.IGNORECASE | re.VERBOSE
)

CASE_SENSITIVE = False

MIN_PAGES = 4

# Scan strategy:
# - First N pages (fast “intro/issue framing”)
# - PLUS last M pages (often where conclusions/reasons live)
TEXT_PAGES_HEAD = 12
TEXT_PAGES_TAIL = 6

MAX_WORKERS = 24
PRESERVE_STRUCTURE = True

# Single CSV written ONLY in MATCHES_ROOT
MASTER_CSV_NAME = "_matches_index.csv"

# Chunking to avoid submitting 200k futures at once
SUBMIT_CHUNK_SIZE = 2000

# Regex-only folders
REGEX_APPEAL_FOLDER_NAME = "_APPEAL_SCOPE_REGEX"
REGEX_INTENT_FOLDER_NAME = "_ASSUMED_INTENTION_REGEX"
REGEX_ONLY_FOLDER_NAME = "_REGEX_ONLY"  # matched by regex but no NEEDLES_ANY substring hits

# =========================================================
# HELPERS
# =========================================================

def _norm(s: str) -> str:
    return s if CASE_SENSITIVE else s.lower()


def _slugify(s: str) -> str:
    s = (s or "").strip().replace(" ", "_")
    s = "".join(ch for ch in s if ch.isalnum() or ch in ("_", "-", "."))
    return s[:120] if s else "EMPTY"


def iter_pdfs(root: Path):
    for p in root.rglob("*.pdf"):
        if p.is_file():
            yield p


def _safe_copy(src: Path, dst: Path) -> Path:
    """
    Copy src to dst. If dst exists, add suffix _1, _2, ...
    Returns final path.
    """
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy2(src, dst)
        return dst

    stem, suffix = dst.stem, dst.suffix
    i = 1
    while True:
        cand = dst.with_name(f"{stem}_{i}{suffix}")
        if not cand.exists():
            shutil.copy2(src, cand)
            return cand
        i += 1


def copy_to_subfolder(src: Path, in_root: Path, subfolder: Path, preserve_structure: bool = True) -> Path:
    """
    Copy one PDF into subfolder, optionally preserving structure.
    Returns final copied path.
    """
    if preserve_structure:
        try:
            rel = src.relative_to(in_root)
        except ValueError:
            rel = Path(src.name)
        dst = subfolder / rel
    else:
        dst = subfolder / src.name

    return _safe_copy(src, dst)


def _extract_text_head_tail(doc: fitz.Document, head: int, tail: int) -> str:
    pages = doc.page_count
    head_n = min(max(head, 0), pages)
    tail_n = min(max(tail, 0), pages)

    idxs = list(range(head_n))

    # tail pages (avoid duplicates)
    if tail_n > 0:
        start = max(pages - tail_n, 0)
        tail_idxs = list(range(start, pages))
        for i in tail_idxs:
            if i not in idxs:
                idxs.append(i)

    chunks = []
    for i in idxs:
        try:
            chunks.append(doc.load_page(i).get_text("text"))
        except Exception:
            pass

    return "\n".join(chunks)


def scan_one(pdf_path: str):
    """
    One-pass scan:
      - page count filter
      - extract head+tail text
      - require NEEDLES_ALL (all must be present)
      - compute which NEEDLES_ANY are present (simple substring)
      - detect appeal-scope limitation via APPEAL_SCOPE_REGEX
      - detect assumed intention / motive via ASSUMED_INTENTION_REGEX

    Returns a dict:
      - if matched: { ... matched fields ..., ok=True, error=False }
      - if not matched: None
      - if error: { ok=False, error=True, error_msg=..., path=... }
    """
    p = Path(pdf_path)
    try:
        stat = p.stat()
        size_mb = stat.st_size / (1024 * 1024)

        doc = fitz.open(p)
        pages = doc.page_count
        if pages < MIN_PAGES:
            doc.close()
            return None

        text = _extract_text_head_tail(doc, TEXT_PAGES_HEAD, TEXT_PAGES_TAIL)
        doc.close()

        text_n = _norm(text)

        needles_all = [_norm(x) for x in NEEDLES_ALL if x and x.strip()]
        needles_any = [_norm(x) for x in NEEDLES_ANY if x and x.strip()]

        # 1) MUST satisfy NEEDLES_ALL (gate)
        ok_all = all(k in text_n for k in needles_all) if needles_all else True
        if not ok_all:
            return None

        # 2) substring hits
        hit_any = [k for k in needles_any if k in text_n]

        # 3) regex hits (on raw text; IGNORECASE set)
        m = APPEAL_SCOPE_REGEX.search(text)
        appeal_scope_hit = bool(m)
        appeal_scope_match = (m.group(0)[:250] if m else "")

        mi = ASSUMED_INTENTION_REGEX.search(text)
        assumed_intention_hit = bool(mi)
        assumed_intention_match = (mi.group(0)[:250] if mi else "")

        # keep doc if it hits at least one NEEDLES_ANY OR either regex hit
        if (not hit_any) and (not appeal_scope_hit) and (not assumed_intention_hit):
            return None

        return {
            "ok": True,
            "error": False,

            "path": str(p),
            "pages": int(pages),
            "size_mb": round(size_mb, 3),
            "mtime": pd.to_datetime(stat.st_mtime, unit="s"),

            "hit_all": "; ".join([k for k in needles_all if k in text_n]),
            "hit_any": "; ".join(hit_any),

            "appeal_scope_hit": bool(appeal_scope_hit),
            "appeal_scope_match": appeal_scope_match,

            "assumed_intention_hit": bool(assumed_intention_hit),
            "assumed_intention_match": assumed_intention_match,

            # convenience
            "regex_only": bool((not hit_any) and (appeal_scope_hit or assumed_intention_hit)),
        }

    except Exception as e:
        return {
            "ok": False,
            "error": True,
            "error_msg": f"{type(e).__name__}: {str(e)[:300]}",
            "path": str(p),
        }


def _submit_in_chunks(executor: ProcessPoolExecutor, items, chunk_size: int):
    """
    Yield futures, but avoid submitting everything at once (RAM-friendly).
    """
    chunk = []
    for it in items:
        chunk.append(it)
        if len(chunk) >= chunk_size:
            for p in chunk:
                yield executor.submit(scan_one, str(p))
            chunk = []
    for p in chunk:
        yield executor.submit(scan_one, str(p))


# =========================================================
# MAIN
# =========================================================

def main():
    MATCHES_ROOT.mkdir(parents=True, exist_ok=True)

    pdfs = list(iter_pdfs(INPUT_ROOT))
    rows = []
    stats = Counter()

    print(f"[scan] Input root: {INPUT_ROOT}")
    print(f"[scan] PDFs found: {len(pdfs)}")
    print(f"[scan] NEEDLES_ALL (must match all): {NEEDLES_ALL}")
    print(f"[scan] NEEDLES_ANY (substring OR regex): {NEEDLES_ANY}")
    print(f"[scan] Pages >= {MIN_PAGES}")
    print(f"[scan] Text scan: head={TEXT_PAGES_HEAD} pages, tail={TEXT_PAGES_TAIL} pages")
    print(f"[scan] Workers={MAX_WORKERS} | submit_chunk={SUBMIT_CHUNK_SIZE}")
    print(f"[out] Principal matches folder: {MATCHES_ROOT}")
    print(f"[out] Preserve structure: {PRESERVE_STRUCTURE}")

    # 1) Scan once in parallel (chunked submission)
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = list(_submit_in_chunks(ex, pdfs, SUBMIT_CHUNK_SIZE))

        for fut in tqdm(as_completed(futs), total=len(futs), desc="Scanning PDFs", unit="file"):
            r = fut.result()
            if r is None:
                stats["no_match"] += 1
                continue
            if r.get("error"):
                stats["error"] += 1
                rows.append(r)  # keep error rows for audit
                continue

            stats["match"] += 1
            if r.get("regex_only"):
                stats["match_regex_only"] += 1
            if r.get("appeal_scope_hit"):
                stats["match_appeal_scope_regex"] += 1
            if r.get("assumed_intention_hit"):
                stats["match_assumed_intention_regex"] += 1
            if (r.get("hit_any") or "").strip():
                stats["match_substring_any"] += 1

            rows.append(r)

    df = pd.DataFrame(rows)
    if df.empty:
        print("[scan] Matches: 0 (and no error rows)")
        return df

    # Split out errors so they don't pollute the match routing
    df_err = df[df.get("error", False) == True].copy() if "error" in df.columns else df.iloc[0:0].copy()
    df_ok = df[df.get("ok", False) == True].copy() if "ok" in df.columns else df.iloc[0:0].copy()

    print(
        f"[stats] scanned={len(pdfs)} | match={stats['match']} | no_match={stats['no_match']} | error={stats['error']}"
    )
    print(
        f"[stats] substring_any={stats['match_substring_any']} | appeal_scope_regex={stats['match_appeal_scope_regex']} | "
        f"assumed_intention_regex={stats['match_assumed_intention_regex']} | regex_only={stats['match_regex_only']}"
    )

    if df_ok.empty:
        print("[scan] No OK matches (only errors). Writing CSV only.")
    else:
        df_ok = df_ok.sort_values(["pages", "size_mb"], ascending=False).reset_index(drop=True)
        print(f"[scan] OK matches: {len(df_ok)}")

    # 2) Create one subfolder per NEEDLES_ANY term and copy files into each
    needles_any_norm = [_norm(x) for x in NEEDLES_ANY if x and x.strip()]
    norm_to_original = {_norm(x): x for x in NEEDLES_ANY if x and x.strip()}

    if not df_ok.empty:
        # For routing, explode hit_any into a list (may be empty if only regex hit)
        df_ok["hit_any_list"] = df_ok.get("hit_any", "").fillna("").apply(
            lambda s: [x.strip() for x in s.split(";") if x.strip()]
        )

        for needle_norm in needles_any_norm:
            label = norm_to_original[needle_norm]
            folder_name = _slugify(label)
            out_dir = (MATCHES_ROOT / folder_name).resolve()
            out_dir.mkdir(parents=True, exist_ok=True)

            mask = df_ok["hit_any_list"].apply(lambda xs: needle_norm in xs)
            df_group = df_ok[mask]

            if df_group.empty:
                print(f"[group] '{label}' -> 0 matches (skip)")
                continue

            print(f"[group] '{label}' -> {len(df_group)} matches -> {out_dir}")

            for src_str in tqdm(df_group["path"].tolist(), desc=f"Copying -> {folder_name}", unit="file"):
                src = Path(src_str)
                copy_to_subfolder(
                    src=src,
                    in_root=INPUT_ROOT,
                    subfolder=out_dir,
                    preserve_structure=PRESERVE_STRUCTURE,
                )

        # 2b) Copy ALL appeal-scope regex hits into ONE dedicated folder
        if "appeal_scope_hit" in df_ok.columns:
            df_regex = df_ok[df_ok["appeal_scope_hit"] == True].copy()
        else:
            df_regex = df_ok.iloc[0:0].copy()

        print(f"[regex] APPEAL_SCOPE_REGEX hits: {len(df_regex)}")

        if not df_regex.empty:
            out_dir = (MATCHES_ROOT / REGEX_APPEAL_FOLDER_NAME).resolve()
            out_dir.mkdir(parents=True, exist_ok=True)

            for src_str in tqdm(df_regex["path"].tolist(), desc=f"Copying -> {REGEX_APPEAL_FOLDER_NAME}", unit="file"):
                src = Path(src_str)
                copy_to_subfolder(
                    src=src,
                    in_root=INPUT_ROOT,
                    subfolder=out_dir,
                    preserve_structure=PRESERVE_STRUCTURE,
                )

        # 2c) Copy ALL assumed-intention regex hits into ONE dedicated folder
        if "assumed_intention_hit" in df_ok.columns:
            df_intent = df_ok[df_ok["assumed_intention_hit"] == True].copy()
        else:
            df_intent = df_ok.iloc[0:0].copy()

        print(f"[regex] ASSUMED_INTENTION_REGEX hits: {len(df_intent)}")

        if not df_intent.empty:
            out_dir = (MATCHES_ROOT / REGEX_INTENT_FOLDER_NAME).resolve()
            out_dir.mkdir(parents=True, exist_ok=True)

            for src_str in tqdm(df_intent["path"].tolist(), desc=f"Copying -> {REGEX_INTENT_FOLDER_NAME}", unit="file"):
                src = Path(src_str)
                copy_to_subfolder(
                    src=src,
                    in_root=INPUT_ROOT,
                    subfolder=out_dir,
                    preserve_structure=PRESERVE_STRUCTURE,
                )

        # 2d) Copy regex-only hits into ONE dedicated folder (optional but useful)
        if "regex_only" in df_ok.columns:
            df_regex_only = df_ok[df_ok["regex_only"] == True].copy()
        else:
            df_regex_only = df_ok.iloc[0:0].copy()

        print(f"[regex] REGEX_ONLY (no substring NEEDLES_ANY) hits: {len(df_regex_only)}")

        if not df_regex_only.empty:
            out_dir = (MATCHES_ROOT / REGEX_ONLY_FOLDER_NAME).resolve()
            out_dir.mkdir(parents=True, exist_ok=True)

            for src_str in tqdm(df_regex_only["path"].tolist(), desc=f"Copying -> {REGEX_ONLY_FOLDER_NAME}", unit="file"):
                src = Path(src_str)
                copy_to_subfolder(
                    src=src,
                    in_root=INPUT_ROOT,
                    subfolder=out_dir,
                    preserve_structure=PRESERVE_STRUCTURE,
                )

    # 3) Write ONE master CSV in MATCHES_ROOT (and nowhere else)
    # Combine OK + error rows into one CSV for audit (errors keep error_msg)
    df_out = df.copy()

    # Ensure columns exist
    if "hit_any_list" in df_out.columns:
        df_out = df_out.drop(columns=["hit_any_list"], errors="ignore")

    df_out["matches_root"] = str(MATCHES_ROOT)

    # boolean columns per NEEDLES_ANY (from hit_any)
    needles_any_norm = [_norm(x) for x in NEEDLES_ANY if x and x.strip()]
    norm_to_original = {_norm(x): x for x in NEEDLES_ANY if x and x.strip()}

    for needle_norm in needles_any_norm:
        label = norm_to_original[needle_norm]
        col = f"has__{_slugify(label)}"
        df_out[col] = df_out.get("hit_any", "").fillna("").apply(
            lambda s: needle_norm in [x.strip() for x in s.split(";") if x.strip()]
        )

    # boolean column for appeal-scope regex
    df_out["has__appeal_scope_regex"] = df_out.get("appeal_scope_hit", False).fillna(False).astype(bool)

    # boolean column for assumed-intention regex
    df_out["has__assumed_intention_regex"] = df_out.get("assumed_intention_hit", False).fillna(False).astype(bool)

    # ANY flag (what Moltie would use)
    df_out["has__any_needle"] = (
        df_out[[f"has__{_slugify(x)}" for x in NEEDLES_ANY] + ["has__appeal_scope_regex", "has__assumed_intention_regex"]]
        .fillna(False)
        .any(axis=1)
    )

    master_path = MATCHES_ROOT / MASTER_CSV_NAME
    df_out.to_csv(master_path, index=False, quoting=1)  # csv.QUOTE_ALL = 1
    print(f"[csv] Wrote single master CSV: {master_path}")

    # 4) Optional: print a quick error preview
    if not df_err.empty:
        print("\n[warn] Some PDFs failed parsing/opening. First 10 errors:")
        cols = [c for c in ["path", "error_msg"] if c in df_err.columns]
        print(df_err[cols].head(10).to_string(index=False))

    return df_out


if __name__ == "__main__":
    df = main()
    if isinstance(df, pd.DataFrame) and not df.empty:
        print(df.head(25))

[scan] Input root: /media/hello/Vault/Tribunals/ET_Cases
[scan] PDFs found: 127755
[scan] NEEDLES_ALL (must match all): ['unfair dismissal']
[scan] NEEDLES_ANY (substring OR regex): ['upheld', 'verbal warning', 'no contemporaneous evidence', 'predetermination']
[scan] Pages >= 4
[scan] Text scan: head=12 pages, tail=6 pages
[scan] Workers=24 | submit_chunk=2000
[out] Principal matches folder: /media/hello/Vault/Tribunals/_Matches
[out] Preserve structure: True


Scanning PDFs: 100%|██████████| 127755/127755 [00:26<00:00, 4907.00file/s]


[stats] scanned=127755 | match=5062 | no_match=122692 | error=1
[stats] substring_any=3401 | appeal_scope_regex=261 | assumed_intention_regex=2322 | regex_only=1661
[scan] OK matches: 5062
[group] 'upheld' -> 3145 matches -> /media/hello/Vault/Tribunals/_Matches/upheld


Copying -> upheld: 100%|██████████| 3145/3145 [00:00<00:00, 5642.30file/s]


[group] 'verbal warning' -> 291 matches -> /media/hello/Vault/Tribunals/_Matches/verbal_warning


...

[regex] ASSUMED_INTENTION_REGEX hits: 2322


Copying -> _ASSUMED_INTENTION_REGEX: 100%|██████████| 2322/2322 [00:00<00:00, 3188.39file/s]


[regex] REGEX_ONLY (no substring NEEDLES_ANY) hits: 1661


Copying -> _REGEX_ONLY: 100%|██████████| 1661/1661 [00:00<00:00, 6346.63file/s]
/tmp/ipykernel_274168/2672473033.py:479: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_out["has__appeal_scope_regex"] = df_out.get("appeal_scope_hit", False).fillna(False).astype(bool)
/tmp/ipykernel_274168/2672473033.py:482: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_out["has__assumed_intention_regex"] = df_out.get("assumed_intention_hit", False).fillna(False).astype(bool)


[csv] Wrote single master CSV: /media/hello/Vault/Tribunals/_Matches/_matches_index.csv

[warn] Some PDFs failed parsing/opening. First 10 errors:
                                                                                                           path                                                                                                                                                           error_msg
/media/hello/Vault/Tribunals/ET_Cases/Mrs_L_Shaw_v_BUPA_Care_Homes__BNH__Ltd_2301363-16_and_2302838-16_Full.pdf EmptyFileError: Cannot open empty file: filename='/media/hello/Vault/Tribunals/ET_Cases/Mrs_L_Shaw_v_BUPA_Care_Homes__BNH__Ltd_2301363-16_and_2302838-16_Full.pdf'.
      ok  error                                               path  pages  \
0   True  False  /media/hello/Vault/Tribunals/ET_Cases/Ms_X_Ju_...   15.0   
1   True  False  /media/hello/Vault/Tribunals/ET_Cases/Mrs_S_Me...   36.0   
2   True  False  /media/hello/Vault/Tribunals/ET_Cases/Miss_S_D...   3

In [2]:
import pandas as pd
from pathlib import Path

# =========================================================
# CONFIG
# =========================================================

MATCHES_ROOT = Path(r"/media/hello/Vault/Tribunals/_Matches").resolve()
MASTER_CSV_NAME = "_matches_index.csv"

# =========================================================
# LOAD
# =========================================================

csv_path = MATCHES_ROOT / MASTER_CSV_NAME

df = pd.read_csv(csv_path)

print(f"[info] Total matched cases: {len(df)}")
print(f"[info] Columns: {len(df.columns)}")

# =========================================================
# FIND BOOLEAN MATCH COLUMNS
# =========================================================

match_cols = [c for c in df.columns if c.startswith("has__")]

if not match_cols:
    print("No has__ columns found.")
    raise SystemExit

# Ensure boolean
for c in match_cols:
    df[c] = df[c].fillna(False).astype(bool)

# =========================================================
# FREQUENCY CALCULATION
# =========================================================

freq_rows = []

total = len(df)

for col in match_cols:
    count = df[col].sum()
    pct = round((count / total) * 100, 2) if total > 0 else 0
    freq_rows.append({
        "match_type": col,
        "count": int(count),
        "percentage_of_total": pct
    })

freq_df = pd.DataFrame(freq_rows).sort_values(
    by="count",
    ascending=False
).reset_index(drop=True)

print("\n=== MATCH FREQUENCY TABLE ===\n")
print(freq_df)

# Optional: save
freq_out = MATCHES_ROOT / "_match_frequencies.csv"
freq_df.to_csv(freq_out, index=False)
print(f"\n[csv] Frequency table written to: {freq_out}")

[info] Total matched cases: 5063
[info] Columns: 22

=== MATCH FREQUENCY TABLE ===

                         match_type  count  percentage_of_total
0                   has__any_needle   5062                99.98
1                       has__upheld   3145                62.12
2      has__assumed_intention_regex   2322                45.86
3               has__verbal_warning    291                 5.75
4           has__appeal_scope_regex    261                 5.16
5             has__predetermination     64                 1.26
6  has__no_contemporaneous_evidence     42                 0.83

[csv] Frequency table written to: /media/hello/Vault/Tribunals/_Matches/_match_frequencies.csv
